# 9. Regression Basics — Learning the Tool Before Building the Real System

**Building a Heart Disease Risk-Screening System — Notebook 9 of 12, Stage 5: Building and Validating the Predictive Core**

Stage 4 validated a set of trustworthy inputs. Before assembling the actual
disease-risk system (Notebooks 10-12), this notebook learns the regression tool
itself on a simpler, well-understood relationship within the same registry:
**predicting a patient's max heart rate (`thalach`) from their age** — a
relationship with a known clinical rule of thumb (max heart rate ≈ 220 − age) to
compare a learned model against.

## The topic

Linear regression fits a line (or hyperplane, with more inputs) that predicts a
continuous outcome from one or more inputs, chosen to minimize the total squared
prediction error.

## Why it matters for this system

Every coefficient in a regression model answers a specific, actionable question —
"how much does the prediction change per unit change in this input, holding the
others fixed." That's exactly the mechanism Notebooks 10-12 will reuse (via a
classifier) to build the actual disease-risk system — learning the mechanics here,
on an outcome simple enough to sanity-check against a known formula, is safer than
learning them for the first time on the real clinical stakes.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error

df = pd.read_csv("../5. MLOps/2. End-to-End ML/data/heart_disease_cleaned_2.csv", index_col=0)
X_age = df[["age"]].to_numpy()
y_thalach = df["thalach"].to_numpy()
print(df[["age", "thalach"]].describe().round(1))

## The toolkit

| Tool | Answers |
|---|---|
| **Simple linear regression** | One input predicting one continuous outcome |
| **Multiple regression** | Several inputs together, each with a "holding others fixed" coefficient |
| **R²** | What fraction of the outcome's variation the model explains |
| **Statistical significance (per coefficient)** | Whether an individual input's effect is distinguishable from zero |
| **Residual diagnostics** | Whether a straight line is even the right shape for this relationship |

## How to choose

Start simple — one input, one outcome — to build intuition and get a sanity-check
baseline, exactly as this notebook does with `age` alone. Move to multiple
regression once you have several validated inputs (Stage 4's output) and want to
know each one's *marginal* contribution rather than its relationship in isolation.
Always follow a fitted model with residual diagnostics — R² and p-values can look
fine even when a straight line is fundamentally the wrong shape for the
relationship.

## Applied to the registry

### A known formula to check the model against

Exercise physiology has a well-known rule of thumb: predicted max heart rate ≈
220 − age. Before fitting anything, see how well that simple formula already does.

In [ ]:
formula_prediction = 220 - df["age"]
formula_mae = mean_absolute_error(df["thalach"], formula_prediction)
print(f"'220 - age' formula MAE: {formula_mae:.1f} bpm")

### Fitting the regression

In [ ]:
model = LinearRegression().fit(X_age, y_thalach)
print(f"Fitted: thalach = {model.intercept_:.2f} + {model.coef_[0]:.3f} * age")
print(f"(the textbook formula is implicitly: thalach = 220 + (-1.0) * age)")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df["age"], df["thalach"], alpha=0.4, color="steelblue", label="patients")
xs = np.linspace(df["age"].min(), df["age"].max(), 100).reshape(-1, 1)
ax.plot(xs, model.predict(xs), "r-", lw=2, label="fitted model")
ax.plot(xs, 220 - xs, "k--", lw=2, label="220 - age formula")
ax.set_xlabel("age"); ax.set_ylabel("thalach (max heart rate)"); ax.legend()
plt.show()

In [ ]:
model_predictions = model.predict(X_age)
model_mae = mean_absolute_error(y_thalach, model_predictions)
r2 = r2_score(y_thalach, model_predictions)

print(f"Fitted model MAE: {model_mae:.1f} bpm   (vs. formula's {formula_mae:.1f} bpm)")
print(f"R²: {r2:.3f} -- age alone explains {r2:.0%} of the variation in max heart rate")

### Adding more inputs: does the extra complexity earn its keep?

Extend to a few of Stage 4's validated inputs and see whether R² improves
meaningfully, or whether age alone was already capturing most of what these inputs
have to offer.

In [ ]:
extra_inputs = ["age", "sex", "trestbps", "exang"]
X_multi = df[extra_inputs].to_numpy()
model_multi = LinearRegression().fit(X_multi, y_thalach)
r2_multi = r2_score(y_thalach, model_multi.predict(X_multi))

coef_table = pd.DataFrame({"input": extra_inputs, "coefficient": model_multi.coef_})
print(coef_table.round(3))
print(f"\nR² with age alone:        {r2:.3f}")
print(f"R² with {len(extra_inputs)} inputs: {r2_multi:.3f}")

### Statistical significance of each coefficient

In [ ]:
import statsmodels.api as sm

X_with_const = sm.add_constant(df[extra_inputs])
sm_model = sm.OLS(df["thalach"], X_with_const).fit()
print(sm_model.summary().tables[1])

### Residual diagnostics: is a straight line even right here?

In [ ]:
residuals = y_thalach - model_multi.predict(X_multi)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(model_multi.predict(X_multi), residuals, alpha=0.4, color="darkorange")
axes[0].axhline(0, color="black", lw=1, linestyle="--")
axes[0].set_xlabel("predicted thalach"); axes[0].set_ylabel("residual"); axes[0].set_title("Residuals vs. fitted")
axes[1].hist(residuals, bins=20, color="seagreen", alpha=0.7)
axes[1].set_title("Residual distribution")
plt.tight_layout(); plt.show()

## Systems view — what this stage hands to the next one

The regression mechanics — fit, interpret coefficients, check R², diagnose
residuals — are now proven on a relationship simple enough to sanity-check against
a known formula. Notebooks 10-12 reuse every one of these mechanics for the real
goal: a classifier predicting disease risk itself, this time asking the harder
question of whether the model will hold up on patients it hasn't seen yet.

## Try it yourself

1. Add `chol` and `thalach`'s own correlate `cp` to the multi-input model — does R²
   improve further, or plateau?
2. Compare the fitted model's predictions against the "220 − age" formula
   specifically for patients over 60 — does the data-driven model diverge from the
   textbook formula more or less at the extremes of the age range?
3. Using `sm_model.conf_int()`, find the 95% confidence interval for the `age`
   coefficient — how does its width compare to the coefficient's own size?